# Building a Honeypot

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:
import os
import re
import subprocess
import time
from collections import Counter

def setup_cowrie():
    """Sets up Cowrie honeypot by installing dependencies and configuring it."""
    print("Installing dependencies...")
    subprocess.run("sudo apt update && sudo apt install git python3-venv python3-dev libssl-dev libffi-dev build-essential -y", shell=True)

    print("Cloning Cowrie repository...")
    if not os.path.exists("cowrie"):
        subprocess.run("git clone https://github.com/cowrie/cowrie.git", shell=True)
    os.chdir("cowrie")

    print("Setting up virtual environment...")
    subprocess.run("python3 -m venv cowrie-env", shell=True)
    subprocess.run("source cowrie-env/bin/activate && pip install --upgrade pip", shell=True, executable="/bin/bash")
    subprocess.run("pip install -r requirements.txt", shell=True)

    print("Configuring Cowrie...")
    if not os.path.exists("cowrie.cfg"):
        subprocess.run("cp cowrie.cfg.dist cowrie.cfg", shell=True)

    print("Starting Cowrie honeypot...")
    subprocess.run("bin/cowrie start", shell=True)

def analyze_logs(log_file="cowrie/var/log/cowrie/cowrie.log"):
    """Parses Cowrie logs to extract attack patterns."""
    if not os.path.exists(log_file):
        print("Log file not found. Waiting for attacks...")
        return

    with open(log_file, "r") as f:
        logs = f.readlines()

    login_attempts = []
    commands_executed = []

    for line in logs:
        if "login attempt" in line:
            ip_match = re.search(r'(\d+\.\d+\.\d+\.\d+)', line)
            if ip_match:
                login_attempts.append(ip_match.group(1))

        if "CMD" in line:
            cmd_match = re.search(r'CMD: (.+)', line)
            if cmd_match:
                commands_executed.append(cmd_match.group(1))

    print("\nTop Attacking IPs:")
    for ip, count in Counter(login_attempts).most_common(5):
        print(f"{ip}: {count} attempts")

    print("\nMost Executed Commands:")
    for cmd, count in Counter(commands_executed).most_common(5):
        print(f"{cmd}: {count} times")

def simulate_attack():
    """Simulates an SSH attack attempt to generate logs."""
    print("Simulating an attack...")
    subprocess.run("ssh root@127.0.0.1 -p 2222", shell=True)
    time.sleep(2)

def main():
    setup_cowrie()
    print("Waiting for attacks... Run the script again later to analyze logs.")
    simulate_attack()
    analyze_logs()

if __name__ == "__main__":
    main()
